# Data cleaning

In [8]:
from pathlib import Path
import pandas as pd

In [9]:
data_path = Path("..") / "data/raw/training_pop30_genres.parquet"
df = pd.read_parquet(data_path)
print(f"Loaded {len(df):,} tracks from {data_path}")

Loaded 186,960 tracks from /Users/peppedilillo/Dropbox/Progetti/ml/data/training_pop50_genres.parquet


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 186960 entries, 0 to 186959
Data columns (total 34 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   track_rowid             186960 non-null  int64  
 1   track_name              186960 non-null  object 
 2   artist_name             186960 non-null  object 
 3   artist_rowid            186960 non-null  int64  
 4   album_rowid             186960 non-null  int64  
 5   album_name              186960 non-null  object 
 6   album_type              186960 non-null  object 
 7   label                   186960 non-null  object 
 8   release_date            186960 non-null  object 
 9   release_date_precision  186960 non-null  object 
 10  id_isrc                 186947 non-null  object 
 11  id_upc                  186297 non-null  object 
 12  time_signature          186960 non-null  int64  
 13  tempo                   186960 non-null  float64
 14  key                 

## Fixing NaNs

In [11]:
100 * (missing:=df.isna().sum())[missing > 0] / len(df), missing[missing > 0]

(id_isrc           0.006953
 id_upc            0.354621
 artist_genres    29.322315
 dtype: float64,
 id_isrc             13
 id_upc             663
 artist_genres    54821
 dtype: int64)

Drops `id_upc` column, and rows with NaN `id_isrc`.

In [12]:
df = df.drop("id_upc", axis="columns")
df = df.dropna(subset=("id_isrc",)).reset_index(drop=True)

In [6]:
100 * (missing:=df.isna().sum())[missing > 0] / len(df), missing[missing > 0]

(artist_genres    29.324354
 dtype: float64,
 artist_genres    54821
 dtype: int64)

Fills missing genres with standard token.

In [9]:
df["artist_genres"] = df["artist_genres"].fillna(value="<GENRE_UNKNOWN>")

In [10]:
100 * missing[(missing := df.isna().sum()) > 0] / len(df), missing[missing > 0]

(Series([], dtype: float64), Series([], dtype: int64))

## Data types

In [11]:
df

,track_rowid,track_name,artist_name,artist_rowid,album_rowid,album_name,album_type,label,release_date,release_date_precision,...,explicit,track_popularity,artist_popularity,artist_followers,album_popularity,duration_ms,track_number,disc_number,total_tracks,artist_genres
0,5,I Just Missed A Call,NOTD,1853555,2,Digital Notes,single,Universal Music AB,2025-03-14,day,...,0,51,62,317626,42,146424,4,1,6,<GENRE_UNKNOWN>
1,28,IT girl,JADE,5196797,15,FUFN (Fuck You For Now),single,RCA Records Label,2025-03-14,day,...,1,51,64,226527,51,153718,3,1,5,<GENRE_UNKNOWN>
2,129,Betrayal,Warren Zeiders,219151,37,"Relapse, Lies, & Betrayal",album,Warner Records,2025-03-14,day,...,0,51,75,934909,53,180282,3,1,21,country
3,138,Withdrawal,Warren Zeiders,219151,37,"Relapse, Lies, & Betrayal",album,Warner Records,2025-03-14,day,...,0,51,75,934909,53,185680,12,1,21,country
4,204,Afterall,Saveus,4490251,62,Afterall,single,The Bank Records,2025-03-14,day,...,0,51,47,40479,30,202688,1,1,1,dansktop
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186955,2779,Not Like Us,Kendrick Lamar,4765104,358,Not Like Us,single,"Kendrick Lamar, under exclusive license to Int...",2024-05-04,day,...,1,96,97,39723389,85,274192,1,1,1,hip hop | west coast hip hop
186956,2985,BAILE INoLVIDABLE,Bad Bunny,2713710,373,DeBÍ TiRAR MáS FOToS,album,Rimas Entertainment LLC.,2025-01-05,day,...,1,96,100,93195263,100,367725,3,1,17,latin | reggaeton | trap latino | urbano latino
186957,269,BIRDS OF A FEATHER,Billie Eilish,842869,102,HIT ME HARD AND SOFT,album,Darkroom/Interscope Records,2024-05-17,day,...,0,98,95,109602603,94,210373,4,1,10,<GENRE_UNKNOWN>
186958,2998,DtMF,Bad Bunny,2713710,373,DeBÍ TiRAR MáS FOToS,album,Rimas Entertainment LLC.,2025-01-05,day,...,1,98,100,93195263,100,237117,16,1,17,latin | reggaeton | trap latino | urbano latino


In [20]:
df["mode"].unique()

array([1, 0])

In [17]:
for c in df.select_dtypes(include='number').columns:
    print(f"{c} {" " * (20 - len(c))} {df[c].dtype} \t {df[c].min():.2f} \t {df[c].max():.2f}")

track_rowid           int64 	 1.00 	 255162133.00
artist_rowid          int64 	 95.00 	 14296919.00
album_rowid           int64 	 1.00 	 58358798.00
time_signature        int64 	 0.00 	 5.00
tempo                 float64 	 0.00 	 243.37
key                   int64 	 0.00 	 11.00
mode                  int64 	 0.00 	 1.00
danceability          float64 	 0.00 	 0.99
energy                float64 	 0.00 	 1.00
loudness              float64 	 -54.38 	 4.92
speechiness           float64 	 0.00 	 0.97
acousticness          float64 	 0.00 	 1.00
instrumentalness      float64 	 0.00 	 1.00
liveness              float64 	 0.00 	 1.00
valence               float64 	 0.00 	 1.00
explicit              int64 	 0.00 	 1.00
track_popularity      int64 	 51.00 	 100.00
artist_popularity     int64 	 0.00 	 100.00
artist_followers      int64 	 0.00 	 141174367.00
album_popularity      int64 	 0.00 	 100.00
duration_ms           int64 	 30001.00 	 4500036.00
track_number          int64 	 1.00 	 198.00
dis

Set data types.

In [21]:
df = df.astype({
    # Identifiers
    'track_rowid': 'int64',
    'artist_rowid': 'int64',
    'album_rowid': 'int64',
    
    # Strings
    'track_name': 'string',
    'artist_name': 'string',
    'album_name': 'string',
    'label': 'string',
    'release_date': 'string',
    'id_isrc': 'string',
    'artist_genres': 'string',
    
    # Categoricals
    'album_type': 'category',
    'release_date_precision': 'category',
    
    # Audio features
    'tempo': 'float32',
    'danceability': 'float32',
    'energy': 'float32',
    'loudness': 'float32',
    'speechiness': 'float32',
    'acousticness': 'float32',
    'instrumentalness': 'float32',
    'liveness': 'float32',
    'valence': 'float32',
    
    # Small integers
    'time_signature': 'uint8',
    'key': 'uint8',
    'mode': 'uint8',
    'explicit': 'bool',
    'track_popularity': 'uint8',
    'artist_popularity': 'uint8',
    'album_popularity': 'uint8',
    'track_number': 'int16',
    'disc_number': 'uint8',
    'total_tracks': 'int16',
    
    # Larger integers
    'artist_followers': 'int64',
    'duration_ms': 'int32',
})

In [22]:
for c in df.select_dtypes(include='number').columns:
    print(f"{c} {" " * (20 - len(c))} {df[c].dtype} \t {df[c].min():.2f} \t {df[c].max():.2f}")

track_rowid           int64 	 1.00 	 255162133.00
artist_rowid          int64 	 95.00 	 14296919.00
album_rowid           int64 	 1.00 	 58358798.00
time_signature        uint8 	 0.00 	 5.00
tempo                 float32 	 0.00 	 243.37
key                   uint8 	 0.00 	 11.00
mode                  uint8 	 0.00 	 1.00
danceability          float32 	 0.00 	 0.99
energy                float32 	 0.00 	 1.00
loudness              float32 	 -54.38 	 4.92
speechiness           float32 	 0.00 	 0.97
acousticness          float32 	 0.00 	 1.00
instrumentalness      float32 	 0.00 	 1.00
liveness              float32 	 0.00 	 1.00
valence               float32 	 0.00 	 1.00
track_popularity      uint8 	 51.00 	 100.00
artist_popularity     uint8 	 0.00 	 100.00
artist_followers      int64 	 0.00 	 141174367.00
album_popularity      uint8 	 0.00 	 100.00
duration_ms           int32 	 30001.00 	 4500036.00
track_number          int16 	 1.00 	 198.00
disc_number           uint8 	 1.00 	 18.00
to